Важное уточнение: модель ниже не предсказывает отдельно категорию `высокий риск`. Мы сводим задачу к бинарной классификации:

- `0` — низкий риск;
- `1` — средний или высокий риск.

Поэтому результат модели называется `model_attention_probability`, то есть вероятность того, что класс требует внимания.

## 1. Импорты и загрузка данных

Используем `pandas` для работы с таблицей и `scikit-learn` для обучения моделей.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score,
)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

DATA_PATH = Path("synthetic_class_rating_data_v2.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("/mnt/data/synthetic_class_rating_data_v2.csv")

df = pd.read_csv(DATA_PATH)

df.head()

,academic_year,class_name,grade,class_letter,students_count,avg_grade_5,attendance_pct,homework_completion_pct,tests_avg_pct,behavior_score_100,olympiad_participation_pct,club_participation_pct,parent_engagement_pct,exam_grade_flag,rating_score_100,rating_place,rating_group,risk_probability,risk_score_100,risk_level,risk_reason
0,2025/2026,1А,1,А,25,3.86,96.0,88.9,76.6,83.3,20.5,71.5,68.3,0,79.1,11,B: сильный класс,0.235,23.5,низкий,критических факторов нет
1,2025/2026,1Б,1,Б,25,3.69,95.2,75.9,75.3,83.9,20.3,61.7,75.5,0,75.3,24,C: стабильный класс,0.314,31.4,средний,умеренный риск по совокупности факторов
2,2025/2026,1В,1,В,31,3.88,94.8,83.4,77.1,94.0,16.0,59.2,72.7,0,79.1,12,B: сильный класс,0.219,21.9,низкий,критических факторов нет
3,2025/2026,1Г,1,Г,23,3.99,98.1,90.8,88.6,99.0,15.9,63.2,73.7,0,85.5,1,A: лидер,0.114,11.4,низкий,критических факторов нет
4,2025/2026,1Д,1,Д,26,3.67,89.8,82.1,65.7,83.1,15.1,57.4,79.6,0,72.7,34,C: стабильный класс,0.381,38.1,средний,посещаемость < 90%; средний результат контроль...


## 2. Быстрая проверка датасета

Проверяем размер таблицы, список полей и распределение уровней риска. Это нужно, чтобы понимать, насколько задача сбалансирована.

In [2]:
print("Размер датасета:", df.shape)
print("\nКолонки:")
print(df.columns.tolist())

print("\nРаспределение risk_level:")
print(df["risk_level"].value_counts())

Размер датасета: (66, 21)

Колонки:
['academic_year', 'class_name', 'grade', 'class_letter', 'students_count', 'avg_grade_5', 'attendance_pct', 'homework_completion_pct', 'tests_avg_pct', 'behavior_score_100', 'olympiad_participation_pct', 'club_participation_pct', 'parent_engagement_pct', 'exam_grade_flag', 'rating_score_100', 'rating_place', 'rating_group', 'risk_probability', 'risk_score_100', 'risk_level', 'risk_reason']

Распределение risk_level:
risk_level
средний    36
низкий     19
высокий    11
Name: count, dtype: int64


## 3. Постановка ML-задачи

Для первого прототипа используем бинарную классификацию.

Целевая переменная:

```text
risk_target = 1, если risk_level = "средний" или "высокий"
risk_target = 0, если risk_level = "низкий"
```

Такой вариант устойчивее для маленького датасета, чем попытка сразу предсказывать три категории риска.

In [3]:
df["risk_target"] = df["risk_level"].isin(["средний", "высокий"]).astype(int)

print(df["risk_target"].value_counts())

df[["class_name", "risk_level", "risk_target"]].head(10)

risk_target
1    47
0    19
Name: count, dtype: int64


,class_name,risk_level,risk_target
0,1А,низкий,0
1,1Б,средний,1
2,1В,низкий,0
3,1Г,низкий,0
4,1Д,средний,1
5,1Е,высокий,1
6,2А,низкий,0
7,2Б,средний,1
8,2В,низкий,0
9,2Г,низкий,0


## 4. Выбор признаков и защита от утечки данных

В модель включаем только те признаки, которые могли бы быть известны до расчёта риска.

Не используем:

- `risk_probability`;
- `risk_score_100`;
- `risk_level`;
- `risk_reason`;
- `rating_place`;
- `rating_group`.

Эти поля либо напрямую описывают риск, либо являются итоговыми аналитическими полями. Если дать их модели, возникнет утечка данных.

In [4]:
feature_cols = [
    "grade",
    "students_count",
    "avg_grade_5",
    "attendance_pct",
    "homework_completion_pct",
    "tests_avg_pct",
    "behavior_score_100",
    "olympiad_participation_pct",
    "club_participation_pct",
    "parent_engagement_pct",
    "exam_grade_flag",
]

target_col = "risk_target"

X = df[feature_cols].copy()
y = df[target_col].copy()

print("Признаки модели:")
print(feature_cols)

print("\nПропуски в признаках:")
print(X.isna().sum())

X.head()

Признаки модели:
['grade', 'students_count', 'avg_grade_5', 'attendance_pct', 'homework_completion_pct', 'tests_avg_pct', 'behavior_score_100', 'olympiad_participation_pct', 'club_participation_pct', 'parent_engagement_pct', 'exam_grade_flag']

Пропуски в признаках:
grade                         0
students_count                0
avg_grade_5                   0
attendance_pct                0
homework_completion_pct       0
tests_avg_pct                 0
behavior_score_100            0
olympiad_participation_pct    0
club_participation_pct        0
parent_engagement_pct         0
exam_grade_flag               0
dtype: int64


,grade,students_count,avg_grade_5,attendance_pct,homework_completion_pct,tests_avg_pct,behavior_score_100,olympiad_participation_pct,club_participation_pct,parent_engagement_pct,exam_grade_flag
0,1,25,3.86,96.0,88.9,76.6,83.3,20.5,71.5,68.3,0
1,1,25,3.69,95.2,75.9,75.3,83.9,20.3,61.7,75.5,0
2,1,31,3.88,94.8,83.4,77.1,94.0,16.0,59.2,72.7,0
3,1,23,3.99,98.1,90.8,88.6,99.0,15.9,63.2,73.7,0
4,1,26,3.67,89.8,82.1,65.7,83.1,15.1,57.4,79.6,0


## 5. Логистическая регрессия

Используем логистическую регрессию. Модель оценивает вероятность того, что класс попадёт в зону внимания:

$$
P(y_i = 1 \mid X_i) = \frac{1}{1 + e^{-z_i}}
$$

где:

$$
z_i = \beta_0\\
+ \beta_1 \cdot grade_i\\
+ \beta_2 \cdot students\_count_i\\
+ \beta_3 \cdot avg\_grade\_5_i\\
+ \beta_4 \cdot attendance\_pct_i\\
+ \beta_5 \cdot homework\_completion\_pct_i\\
+ \beta_6 \cdot tests\_avg\_pct_i\\
+ \beta_7 \cdot behavior\_score\_100_i\\
+ \beta_8 \cdot olympiad\_participation\_pct_i\\
+ \beta_9 \cdot club\_participation\_pct_i\\
+ \beta_{10} \cdot parent\_engagement\_pct_i\\
+ \beta_{11} \cdot exam\_grade\_flag_i\\
$$

В коде перед логистической регрессией используется `StandardScaler`, поэтому фактически модель обучается на стандартизированных признаках:

$$
x_{scaled} = \frac{x - mean(x)}{std(x)}
$$

Это нужно, чтобы признаки в разных шкалах корректно сравнивались между собой.

## 6. Сравнение моделей

Сравниваем несколько вариантов:

1. `DummyClassifier` - базовая модель, чтобы понимать минимальный уровень качества.
2. `LogisticRegression` - основная модель для прототипа.
3. `DecisionTreeClassifier` - простая интерпретируемая альтернатива.
4. `RandomForestClassifier` - более сильная, но менее прозрачная модель.

Для риска особенно важен `recall`: лучше лишний раз обратить внимание на класс, чем пропустить реальную проблему.

In [5]:
models = {
    "dummy_baseline": DummyClassifier(strategy="most_frequent"),

    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=42,
        )),
    ]),

    "decision_tree_depth_3": DecisionTreeClassifier(
        max_depth=3,
        class_weight="balanced",
        random_state=42,
    ),

    "random_forest_depth_3": RandomForestClassifier(
        n_estimators=200,
        max_depth=3,
        class_weight="balanced",
        random_state=42,
    ),
}

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42,
)

scoring = {
    "roc_auc": "roc_auc",
    "average_precision": "average_precision",
    "balanced_accuracy": "balanced_accuracy",
    "f1": "f1",
    "recall": "recall",
    "precision": "precision",
}

results = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X,
        y,
        cv=cv,
        scoring=scoring,
        error_score="raise",
    )

    row = {"model": model_name}
    for metric_name in scoring:
        values = scores[f"test_{metric_name}"]
        row[metric_name] = round(values.mean(), 3)
        row[f"{metric_name}_std"] = round(values.std(), 3)

    results.append(row)

results_df = pd.DataFrame(results)
results_df.sort_values("roc_auc", ascending=False)

,model,roc_auc,roc_auc_std,average_precision,average_precision_std,balanced_accuracy,balanced_accuracy_std,f1,f1_std,recall,recall_std,precision,precision_std
1,logistic_regression,0.994,0.011,0.998,0.004,0.928,0.092,0.956,0.039,0.956,0.054,0.964,0.073
3,random_forest_depth_3,0.994,0.011,0.998,0.004,0.890,0.140,0.950,0.052,0.980,0.040,0.930,0.098
2,decision_tree_depth_3,0.814,0.094,0.880,0.069,0.814,0.094,0.894,0.032,0.896,0.063,0.904,0.086
0,dummy_baseline,0.500,0.000,0.712,0.030,0.500,0.000,0.831,0.020,1.000,0.000,0.712,0.030


## 7. Обучение финальной модели

Для финального прототипа выбираем логистическую регрессию, потому что она:

- возвращает вероятность;
- хорошо объясняется через коэффициенты;
- подходит для небольшого датасета;
- прозрачнее случайного леса и нейросетей.

В реальном проекте модель нужно будет обучать на исторических данных и проверять на новых периодах.

In [6]:
final_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

final_model.fit(X, y)

final_model

Pipeline(steps=[('scaler', StandardScaler()),
                ('model',
                 LogisticRegression(class_weight='balanced', max_iter=1000,
                                    random_state=42))])

## 8. Проверка на отложенной выборке

Из-за маленького размера датасета одиночный train/test split может быть нестабильным. Поэтому основная оценка выше - кросс-валидация. Но отдельную проверку на отложенной выборке всё равно полезно показать.

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    stratify=y,
    random_state=42,
)

test_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(
        max_iter=1000,
        class_weight="balanced",
        random_state=42,
    )),
])

test_model.fit(X_train, y_train)

y_pred = test_model.predict(X_test)
y_proba = test_model.predict_proba(X_test)[:, 1]

print("ROC-AUC:", round(roc_auc_score(y_test, y_proba), 3))
print("Average precision:", round(average_precision_score(y_test, y_proba), 3))

print("\nClassification report:")
print(classification_report(y_test, y_pred))

print("\nConfusion matrix:")
print(confusion_matrix(y_test, y_pred))

ROC-AUC: 0.983
Average precision: 0.994

Classification report:
              precision    recall  f1-score   support

           0       0.83      1.00      0.91         5
           1       1.00      0.92      0.96        12

    accuracy                           0.94        17
   macro avg       0.92      0.96      0.93        17
weighted avg       0.95      0.94      0.94        17


Confusion matrix:
[[ 5  0]
 [ 1 11]]


## 9. Интерпретация коэффициентов

Коэффициенты показывают направление связи признака с вероятностью попадания в зону внимания.

- Положительный коэффициент - при росте признака вероятность зоны внимания увеличивается.
- Отрицательный коэффициент - при росте признака вероятность зоны внимания снижается.

Так как модель обучалась на стандартизированных признаках, сначала смотрим коэффициенты в стандартизированной шкале.

In [8]:
scaler = final_model.named_steps["scaler"]
log_reg = final_model.named_steps["model"]

coef_scaled_df = pd.DataFrame({
    "feature": feature_cols,
    "coef_scaled": log_reg.coef_[0],
})

coef_scaled_df["abs_coef_scaled"] = coef_scaled_df["coef_scaled"].abs()
coef_scaled_df = coef_scaled_df.sort_values("abs_coef_scaled", ascending=False)

print("Intercept для стандартизированных признаков:", round(log_reg.intercept_[0], 4))
coef_scaled_df

Intercept для стандартизированных признаков: 2.4728


,feature,coef_scaled,abs_coef_scaled
2,avg_grade_5,-1.501683,1.501683
4,homework_completion_pct,-1.257996,1.257996
3,attendance_pct,-1.233858,1.233858
5,tests_avg_pct,-0.783752,0.783752
6,behavior_score_100,-0.744170,0.744170
10,exam_grade_flag,0.705173,0.705173
1,students_count,-0.426214,0.426214
9,parent_engagement_pct,-0.171032,0.171032
8,club_participation_pct,0.058795,0.058795
7,olympiad_participation_pct,-0.025735,0.025735


## 10. Формула модели в исходных единицах

Для документации можно пересчитать коэффициенты из стандартизированной шкалы обратно в исходные единицы признаков. Это делает формулу более понятной для описания проекта.

In [9]:
coef_original = log_reg.coef_[0] / scaler.scale_
intercept_original = log_reg.intercept_[0] - np.sum(
    log_reg.coef_[0] * scaler.mean_ / scaler.scale_
)

formula_original_df = pd.DataFrame({
    "feature": feature_cols,
    "coef_original_units": coef_original,
})

formula_original_df["abs_coef_original_units"] = formula_original_df["coef_original_units"].abs()
formula_original_df = formula_original_df.sort_values("abs_coef_original_units", ascending=False)

print("Intercept в исходных единицах:", round(intercept_original, 6))
formula_original_df

Intercept в исходных единицах: 81.999825


,feature,coef_original_units,abs_coef_original_units
2,avg_grade_5,-4.276974,4.276974
10,exam_grade_flag,1.828320,1.828320
3,attendance_pct,-0.303364,0.303364
1,students_count,-0.195892,0.195892
4,homework_completion_pct,-0.174425,0.174425
6,behavior_score_100,-0.116207,0.116207
5,tests_avg_pct,-0.100871,0.100871
9,parent_engagement_pct,-0.020337,0.020337
0,grade,0.006016,0.006016
8,club_participation_pct,0.005964,0.005964


In [10]:
formula_terms = [f"({intercept_original:.6f})"]

for feature, coef in zip(feature_cols, coef_original):
    sign = "+" if coef >= 0 else "-"
    formula_terms.append(f" {sign} {abs(coef):.6f} * {feature}")

formula_text = "z = " + "".join(formula_terms)

print("Формула линейной части модели в исходных единицах:")
print(formula_text)
print("\nP(y = 1 | X) = 1 / (1 + exp(-z))")

Формула линейной части модели в исходных единицах:
z = (81.999825) + 0.006016 * grade - 0.195892 * students_count - 4.276974 * avg_grade_5 - 0.303364 * attendance_pct - 0.174425 * homework_completion_pct - 0.100871 * tests_avg_pct - 0.116207 * behavior_score_100 - 0.004278 * olympiad_participation_pct + 0.005964 * club_participation_pct - 0.020337 * parent_engagement_pct + 1.828320 * exam_grade_flag

P(y = 1 | X) = 1 / (1 + exp(-z))


## 11. Получение вероятности зоны внимания

Модель возвращает вероятность `model_attention_probability`.

Это не вероятность именно высокого риска. Это вероятность того, что класс относится к `среднему` или `высокому` риску, то есть требует внимания.

In [11]:
df["model_attention_probability"] = final_model.predict_proba(X)[:, 1]

df[[
    "class_name",
    "risk_level",
    "risk_target",
    "model_attention_probability",
]].sort_values("model_attention_probability", ascending=False).head(15)

,class_name,risk_level,risk_target,model_attention_probability
65,11Е,высокий,1,0.999999
52,9Д,высокий,1,0.999999
53,9Е,высокий,1,0.999986
63,11Г,высокий,1,0.999980
51,9Г,высокий,1,0.999978
64,11Д,высокий,1,0.999975
58,10Д,высокий,1,0.999973
59,10Е,высокий,1,0.999734
41,7Е,средний,1,0.999715
5,1Е,высокий,1,0.999392


## 12. Перевод вероятности в категорию для дашборда

Для дашборда удобно перевести вероятность в понятную категорию.

Пороги для прототипа:

- меньше `0.30` - низкая зона внимания;
- от `0.30` до `0.60` - средняя зона внимания;
- от `0.60` - высокая зона внимания.

В реальном продукте эти пороги нужно будет откалибровать на исторических данных и согласовать с бизнес-заказчиком.

In [12]:
def attention_category(p: float) -> str:
    if p >= 0.60:
        return "высокая зона внимания"
    if p >= 0.30:
        return "средняя зона внимания"
    return "низкая зона внимания"


df["model_attention_level"] = df["model_attention_probability"].apply(attention_category)

df[[
    "class_name",
    "risk_level",
    "model_attention_probability",
    "model_attention_level",
]].sort_values("model_attention_probability", ascending=False).head(20)

,class_name,risk_level,model_attention_probability,model_attention_level
65,11Е,высокий,0.999999,высокая зона внимания
52,9Д,высокий,0.999999,высокая зона внимания
53,9Е,высокий,0.999986,высокая зона внимания
63,11Г,высокий,0.999980,высокая зона внимания
51,9Г,высокий,0.999978,высокая зона внимания
64,11Д,высокий,0.999975,высокая зона внимания
58,10Д,высокий,0.999973,высокая зона внимания
59,10Е,высокий,0.999734,высокая зона внимания
41,7Е,средний,0.999715,высокая зона внимания
5,1Е,высокий,0.999392,высокая зона внимания


## 13. Таблица для дашборда

Формируем итоговую таблицу, которую можно отдать в BI-инструмент или выгрузить в Excel.

In [13]:
dashboard_risk_table = df[[
    "academic_year",
    "class_name",
    "grade",
    "students_count",
    "rating_score_100",
    "risk_level",
    "model_attention_probability",
    "model_attention_level",
    "attendance_pct",
    "homework_completion_pct",
    "tests_avg_pct",
    "behavior_score_100",
    "avg_grade_5",
    "parent_engagement_pct",
    "exam_grade_flag",
]].copy()

dashboard_risk_table = dashboard_risk_table.sort_values(
    "model_attention_probability",
    ascending=False,
)

dashboard_risk_table.head(20)

,academic_year,class_name,grade,students_count,rating_score_100,risk_level,model_attention_probability,model_attention_level,attendance_pct,homework_completion_pct,tests_avg_pct,behavior_score_100,avg_grade_5,parent_engagement_pct,exam_grade_flag
65,2025/2026,11Е,11,22,62.5,высокий,0.999999,высокая зона внимания,81.6,65.5,70.2,70.9,3.00,53.2,1
52,2025/2026,9Д,9,26,60.9,высокий,0.999999,высокая зона внимания,85.0,62.2,57.7,75.0,3.06,58.1,1
53,2025/2026,9Е,9,25,65.1,высокий,0.999986,высокая зона внимания,85.9,67.8,69.1,75.6,3.08,58.3,1
63,2025/2026,11Г,11,27,64.4,высокий,0.999980,высокая зона внимания,88.8,64.4,62.6,80.0,3.08,56.6,1
51,2025/2026,9Г,9,28,63.6,высокий,0.999978,высокая зона внимания,84.1,70.3,60.4,78.8,3.18,57.6,1
64,2025/2026,11Д,11,24,65.7,высокий,0.999975,высокая зона внимания,87.1,67.8,68.3,74.9,3.22,58.5,1
58,2025/2026,10Д,10,25,63.2,высокий,0.999973,высокая зона внимания,87.8,59.9,64.4,75.2,3.11,58.4,0
59,2025/2026,10Е,10,26,65.6,высокий,0.999734,высокая зона внимания,86.9,65.1,67.6,74.7,3.39,59.9,0
41,2025/2026,7Е,7,26,66.5,средний,0.999715,высокая зона внимания,86.1,76.6,70.8,69.5,3.10,50.4,0
5,2025/2026,1Е,1,27,64.9,высокий,0.999392,высокая зона внимания,86.9,70.0,58.0,85.1,3.22,70.9,0


In [14]:
OUTPUT_PATH = Path("dashboard_risk_predictions.xlsx")

dashboard_risk_table.to_excel(OUTPUT_PATH, index=False)

print(f"Файл сохранён: {OUTPUT_PATH.resolve()}")

Файл сохранён: C:\Users\Яна\OneDrive\Desktop\IBS-Dashboard-2026\Data Analysis\dashboard_risk_predictions.xlsx
